<a href="https://colab.research.google.com/github/claramanolache/DeepLearning/blob/main/%5BCS_12B%5D_Module_1_Practice_Activity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Notebook

**You do not need to submit this Notebook.** It is for practice only.

To begin, select "Copy to Drive" at the top (or alternatively download the Notebook file to a local machine for use with Jupyter).

This instance of the Notebook belongs to you, and you are encouraged to make changes to it. Experimentation often leads to new insights and deeper understanding.

If you ever need to revert back to the original version, you can simply make a new copy by accessing it from Canvas again.

### Goals
- Learn about the Tensor data type
- Load data and a model to a GPU
- Run a PyTorch model on MNIST and compare its results with Logistic Regression

In [1]:
import numpy as np

from sklearn import datasets
from sklearn import linear_model

import torch
import torch.nn as nn
from torch.utils import data

Use a seed variable to make random processes deterministic, which can aid with debugging and tuning.

You are free to change this value to explore how the code runs differently.

In [2]:
seed = 12345
np.random.seed(seed)
torch.manual_seed(seed)

For significantly faster training, PyTorch models should be moved to a GPU (Graphics Processing Unit).

When working in Google Colab or locally on a machine with an NVIDIA GPU, the device should be set to `"cuda"`.

If working locally (i.e. not in Google Colab) on a Macbook with Apple silicon or with an AMD GPU, the device should be set to `"mps"`.

Otherwise, the device can be set as `"cpu"`, which means that no GPU acceleration will be available. Note that this option will likely incur significantly longer model training time, so use of Google Colab is recommended if you do not have either of the above hardware options.

In [3]:
device = "cuda"

We can check which options are available using the methods below. If you are running this code on Google Colab and changed the runtime type accordingly, you should see that CUDA is available (and MPS is not).

In [4]:
torch.cuda.is_available()

True

In [5]:
torch.backends.mps.is_available()

False

In general, the following conditional statement can be used to automatically select an appropriate GPU when available.

In [6]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

## Tensors

The fundamental data type in PyTorch is the Tensor, which behaves similarly to a NumPy array. However, it also has other important properties and methods required by PyTorch, and its use in models is required by the library.

In [7]:
X = torch.tensor([[0.0, 10.0, 20.0], [30.0, 40.0, 50.0]])
X

tensor([[ 0., 10., 20.],
        [30., 40., 50.]])

In [8]:
X.shape

torch.Size([2, 3])

In [9]:
X.dtype

torch.float32

### Indexing and Slicing

As with NumPy, dimensions used for indexing or slicing are separated with a comma. The `:` symbol can be used to get all entries in that dimension.

Experiment with different indices to see which values are in the resulting tensor.

In [10]:
X[0, 1]

tensor(10.)

In [11]:
X[:, 1]

tensor([10., 40.])

### Broadcasting and Mathematical Methods

The mathematical rules for NumPy arrays and PyTorch tensors are less strict than formal linear algebra. Here, a scalar value can be added or multiplied to a tensor, and all values in the tensor will have that operation performed on them - a process known as broadcasting.

In [12]:
10 * (X + 1.0)

tensor([[ 10., 110., 210.],
        [310., 410., 510.]])

Again, as with NumPy arrays, there are many mathematical methods available to PyTorch tensors that perform operations to map (apply to every value) or reduce (aggregate the values to a scalar).

In [13]:
X.mean()

tensor(25.)

Aggregations can be performed along a specific dimension (axis) using the `dim` argument.

In [14]:
X.max(dim=0)

torch.return_types.max(
values=tensor([30., 40., 50.]),
indices=tensor([1, 1, 1]))

### Matrix Multiplication

Finally, Python has a matrix multiplication operator `@` that is very useful when working with NumPy arrays and PyTorch tensors.

As with formal linear algebra, the inner dimensions of the matrices need to match. For example, a matrix with dimensions 2x3 can be multiplied by a matrix with dimensions 3x2 because both inner dimensions are 3.

In [15]:
try:
    X @ X
except RuntimeError as e:
    print(e)

mat1 and mat2 shapes cannot be multiplied (2x3 and 2x3)


In [16]:
X.shape, X.T.shape

(torch.Size([2, 3]), torch.Size([3, 2]))

In [17]:
X @ X.T

tensor([[ 500., 1400.],
        [1400., 5000.]])

## Tensor Processing on a GPU

By default, PyTorch tensors will be stored with the CPU (technically, in RAM that the CPU accesses). To see on which device - CPU or GPU - a tensor is stored, use the `device` attribute.

In [18]:
X = torch.rand((1000, 1000))
X

tensor([[0.9817, 0.8796, 0.9921,  ..., 0.5851, 0.0098, 0.8032],
        [0.6510, 0.7297, 0.6304,  ..., 0.5536, 0.0029, 0.1389],
        [0.1970, 0.6486, 0.0293,  ..., 0.2613, 0.2839, 0.6180],
        ...,
        [0.8465, 0.3695, 0.9785,  ..., 0.1686, 0.9599, 0.3065],
        [0.4292, 0.7852, 0.6176,  ..., 0.8840, 0.7409, 0.8170],
        [0.0344, 0.0471, 0.4342,  ..., 0.7951, 0.9992, 0.1963]])

In [19]:
X.device

device(type='cpu')

Before moving this tensor to the GPU, let's see how quickly it can perform multiplication on this 1000x1000 matrix by itself.

In a Colab or Jupyter Notebook, the `%timeit` command displays how long an operation took to complete.

In [20]:
%timeit X @ X.T

71.8 ms ± 34.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


To move the tensor, use the `to` method, specifying `"cuda"` (or `"mps"` for Apple/AMD). Be sure to save the result to a variable, as the original variable will still refer to the tensor with the CPU.

In [21]:
X.to(device)
X.device

device(type='cpu')

In [22]:
X = X.to(device)
X.device

device(type='cuda', index=0)

Now let's see how much faster the same matrix can be multiplied by itself on the GPU. Note that µs refers to a microsecond, which is 1/1000 of a millisecond (ms).

In [23]:
%timeit X @ X.T

516 µs ± 4.11 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


## Logistic Regression

Before running PyTorch, let's create a Logistic Regression model for the MNIST dataset. Recall that MNIST is a collection of 70,000 labeled images - each 28x28 pixels in size - of handwritten digits.

This section will be very brief, as it is assumed you are already familiar with the model and steps used here.

In [24]:
bunch_obj = datasets.fetch_openml("mnist_784", as_frame=False)
X, y = bunch_obj.data, bunch_obj.target
X.shape, y.shape

((70000, 784), (70000,))

Since the minimum pixel value is zero, the operation below is the same as using the `MinMaxScaler` transformer.

In [25]:
X = X / 255.0

The target labels are initially strings; here, they are converted to integers.

In [26]:
y = y.astype(np.int64)

The MNIST dataset is already shuffled, with the final 10,000 samples designed to serve as the test set. For a faster runtime, only a subset of the training and test sets are used for this demonstration.

In [27]:
X_train, X_test, y_train, y_test = X[:10000], X[-1000:], y[:10000], y[-1000:]

In [28]:
clf = linear_model.LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.899

Logistic Regression does quite well on this task. Next, let's see how a feedforward network compares.

## Feedforward Neural Network

This section is only intended to provide a high-level view of processes that will be explained in detail in an upcoming activity.

Additionally, it serves to ensure that you are able to run a model on a GPU (either on Colab or on a local machine).

In [29]:
X_train.shape

(10000, 784)

Each pixel (feature) will serve as one input to the network; thus, there will be 784 (28x28) inputs total.

In [30]:
n_features = X_train.shape[1]
n_features

784

On the other end of the network, the number of unique classes in MNIST determines how many outputs the network requires. As with Logistic Regression, each output (after Softmax is applied) will provide a probability representing the model's confidence that a particular sample belongs to that class.

In [31]:
n_classes = np.unique(y_train)
n_classes

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

As the name implies, feedforward networks are sequential. This model constructor will be unpacked in a later lesson.

In [32]:
model = nn.Sequential(
    nn.Linear(n_features, 128),
    nn.ReLU(),
    nn.Linear(128, len(n_classes))
)

The MNIST data structures downloaded previously are NumPy arrays. As mentioned, PyTorch requires tensors as input. Here, the arrays are converted to tensors.

In [33]:
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train.astype(np.int64))
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test.astype(np.int64))

Additionally, in order to perform mini-batch gradient descent, it is common to use a PyTorch iterator known as a `DataLoader`.

In [34]:
train_dataset = data.TensorDataset(X_train_tensor, y_train_tensor)
train_loader = data.DataLoader(train_dataset, batch_size=32, shuffle=True)

Below, a few hyperparameters for the training process are set. Feel free to increase the number of epochs (iterations) of training, though a high value will require significantly more time to complete.

In [35]:
n_epochs = 50
learning_rate = 0.01
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

### Training the Model

This section contains many steps that work together to train the model. As with the model itself, these steps will be unpacked in an upcoming lesson.

Note that the cell below may require a few minutes to complete; be sure your runtime type is set to a GPU to avoid even longer training times.

In [36]:
model.to(device)
model.train()
for epoch in range(n_epochs):
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    print("Mean Loss:", total_loss / len(train_loader))

Mean Loss: 0.3819908020547785
Mean Loss: 0.18601364359116782
Mean Loss: 0.13211266904141003
Mean Loss: 0.11810990453239519
Mean Loss: 0.10892984466593766
Mean Loss: 0.09069820999801263
Mean Loss: 0.07883194759495545
Mean Loss: 0.07502557710884752
Mean Loss: 0.07448521408513771
Mean Loss: 0.09176992220398188
Mean Loss: 0.05839494421386702
Mean Loss: 0.06192120126452217
Mean Loss: 0.06881659289495806
Mean Loss: 0.05989176774424954
Mean Loss: 0.06653116299307921
Mean Loss: 0.0588807926042998
Mean Loss: 0.06039306882558009
Mean Loss: 0.04957480300559055
Mean Loss: 0.07506857405484413
Mean Loss: 0.05042891621503353
Mean Loss: 0.04608009317003805
Mean Loss: 0.04796653804873358
Mean Loss: 0.05774792711195045
Mean Loss: 0.04902761252140902
Mean Loss: 0.058614512349882616
Mean Loss: 0.05194209993108337
Mean Loss: 0.040219019867828935
Mean Loss: 0.03738536930509117
Mean Loss: 0.048140242604558146
Mean Loss: 0.04235217279246065
Mean Loss: 0.03008247157147598
Mean Loss: 0.03553034960749673
Mean Lo

### Evaluating the Model

As with the training data, the test samples must be wrapped in a `DataLoader` to be able to process it in batches, which will speed up the predictions.

In [37]:
test_dataset = data.TensorDataset(X_test_tensor, y_test_tensor)
test_loader = data.DataLoader(test_dataset, batch_size=32)

The evaluation steps move test batches to the GPU and retrieves the most confident predictions for each sample in the batch.

In [38]:
model.eval()
n_correct = 0
n_samples = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch)
        predicted_labels = torch.argmax(y_pred, dim=1)
        n_correct += (predicted_labels == y_batch).sum().item()
        n_samples += y_batch.size(0)

If everything ran correctly, you should see a significant improvement in the accuracy of the network compared to the Logistic Regression model.

In [39]:
n_correct / n_samples

0.954

## Next Steps

Before we cover the details of the network's training and evaluation phases, you are encouraged to select a different dataset and see how it performs on both the Logistic Regression and network models above.

In an upcoming project, you will be using a feedforward network like this one to perform classification on a dataset of your choice.